# Water Quality Data Analysis & Quality Control
Clean a public water-quality dataset, compare **pH, turbidity and TDS** against standard limits, build **control charts** to flag out-of-range readings, and summarise findings.

**Data:** put your public dataset at `data/water_quality.csv` with columns `sample_id, ph, turbidity, tds` (rename in the *Load* cell if yours differ). If the file is missing, this notebook generates **synthetic demo data** so it still runs end to end.

> Limits below follow common drinking-water guidance (BIS IS 10500 / WHO). Adjust `LIMITS` to the standard you are citing.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LIMITS = {                      # (lower, upper)
    "ph":        (6.5, 8.5),
    "turbidity": (0.0, 5.0),    # NTU
    "tds":       (0.0, 500.0),  # mg/L
}
UNITS = {"ph": "", "turbidity": "NTU", "tds": "mg/L"}

## 1. Load data

In [ ]:
PATH = "data/water_quality.csv"
if os.path.exists(PATH):
    raw = pd.read_csv(PATH)
    raw.columns = raw.columns.str.strip().str.lower()
    print("Loaded real dataset:", raw.shape)
else:
    rng = np.random.default_rng(42)
    n = 300
    raw = pd.DataFrame({
        "sample_id": np.arange(1, n + 1),
        "ph": rng.normal(7.4, 0.5, n),
        "turbidity": rng.gamma(2.0, 1.3, n),
        "tds": rng.normal(380, 90, n),
    })
    # inject typical data problems
    raw.loc[rng.choice(n, 10, replace=False), "ph"] = np.nan
    raw.loc[rng.choice(n, 6, replace=False), "tds"] = np.nan
    raw.loc[5, "ph"] = 41.0            # entry error
    raw.loc[17, "turbidity"] = -3.0    # impossible
    raw = pd.concat([raw, raw.iloc[[3, 4]]], ignore_index=True)  # duplicates
    print("Using SYNTHETIC demo data:", raw.shape)
raw.head()

## 2. Clean

In [ ]:
df = raw.copy()
log = {"rows_in": len(df)}

for c in LIMITS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.drop_duplicates()
log["after_dedupe"] = len(df)

# physically impossible values -> NaN
df.loc[~df["ph"].between(0, 14), "ph"] = np.nan
df.loc[df["turbidity"] < 0, "turbidity"] = np.nan
df.loc[df["tds"] < 0, "tds"] = np.nan

log["missing_before_fill"] = int(df[list(LIMITS)].isna().sum().sum())
for c in LIMITS:
    df[c] = df[c].fillna(df[c].median())   # median is robust to outliers
df = df.reset_index(drop=True)
log["rows_out"] = len(df)
pd.Series(log)

## 3. Compare against standard limits

In [ ]:
rows = []
for c, (lo, hi) in LIMITS.items():
    out = ~df[c].between(lo, hi)
    rows.append({"parameter": c, "unit": UNITS[c], "limit": f"{lo}-{hi}",
                 "mean": df[c].mean(), "min": df[c].min(), "max": df[c].max(),
                 "n_out_of_limit": int(out.sum()), "pct_out": 100 * out.mean()})
compliance = pd.DataFrame(rows).round(2)
df["within_limits"] = np.logical_and.reduce([df[c].between(*LIMITS[c]) for c in LIMITS])
print(f"Samples meeting ALL limits: {df['within_limits'].mean():.1%}")
compliance

## 4. Control charts (Individuals / I-MR)
Centre line = mean. Sigma is estimated from the average moving range (`MRbar / 1.128`). Control limits = mean ± 3σ.
Points are flagged when they fall outside the **control limits** (process shift) or the **specification limits** (non-compliant water).

In [ ]:
def imr_chart(series, name, spec, ax):
    x = series.reset_index(drop=True)
    mr = x.diff().abs()
    sigma = mr.mean() / 1.128
    cl, ucl, lcl = x.mean(), x.mean() + 3 * sigma, x.mean() - 3 * sigma
    ooc = (x > ucl) | (x < lcl)
    oos = (x > spec[1]) | (x < spec[0])
    ax.plot(x, lw=0.8, color="steelblue")
    ax.axhline(cl, color="green"); ax.axhline(ucl, color="red", ls="--"); ax.axhline(lcl, color="red", ls="--")
    ax.axhline(spec[1], color="orange", ls=":"); ax.axhline(spec[0], color="orange", ls=":")
    ax.scatter(x[ooc].index, x[ooc], color="red", zorder=3, label="outside control limits")
    ax.scatter(x[oos].index, x[oos], facecolors="none", edgecolors="orange", s=80, zorder=3, label="outside spec")
    ax.set_title(f"{name} {('('+UNITS[name]+')') if UNITS[name] else ''}  |  UCL={ucl:.2f}  CL={cl:.2f}  LCL={lcl:.2f}")
    ax.legend(loc="upper right", fontsize=8)
    return {"parameter": name, "UCL": ucl, "CL": cl, "LCL": lcl, "out_of_control": int(ooc.sum()), "out_of_spec": int(oos.sum())}

fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
res = [imr_chart(df[c], c, LIMITS[c], ax) for c, ax in zip(LIMITS, axes)]
axes[-1].set_xlabel("Sample order")
plt.tight_layout(); plt.show()
pd.DataFrame(res).round(2)

## 5. Flagged readings

In [ ]:
flag = pd.DataFrame({c: ~df[c].between(*LIMITS[c]) for c in LIMITS})
flagged = df[flag.any(axis=1)].copy()
flagged["failed_parameters"] = flag[flag.any(axis=1)].apply(lambda r: ", ".join(r.index[r]), axis=1)
os.makedirs("outputs", exist_ok=True)
flagged.to_csv("outputs/flagged_samples.csv", index=False)
df.to_csv("outputs/cleaned_water_quality.csv", index=False)
print(len(flagged), "flagged samples saved to outputs/flagged_samples.csv")
flagged.head(10)

## 6. Key findings & recommendations

In [ ]:
worst = compliance.sort_values("pct_out", ascending=False).iloc[0]
print("KEY FINDINGS")
print(f"- {df['within_limits'].mean():.1%} of {len(df)} cleaned samples meet all three limits.")
print(f"- Most frequent exceedance: {worst['parameter']} ({worst['pct_out']:.1f}% of samples outside {worst['limit']}).")
print(f"- {sum(r['out_of_control'] for r in res)} control-chart signals across parameters.")
print("\nRECOMMENDATIONS")
print("- Investigate sampling points/dates behind flagged readings before treatment decisions.")
print("- Re-test samples with high turbidity; filtration/disinfection checks if turbidity is persistently high.")
print("- High TDS: check source mixing and consider RO / blending where needs exceed limits.")
print("- Keep the control chart running on new data and re-baseline limits after process changes.")